# UrbanFlow — Verified Multi-Horizon Test Evaluation

This notebook evaluates the **existing saved checkpoint** at 15, 30, 45, and 60 minutes. It does not train or overwrite the model, original target arrays, or historical result files.

The evaluator verifies target alignment against future test-sequence rows, builds a consistent raw-observation mask from `data/raw/metr-la.h5` (finite and non-zero observations; interpolated values excluded), and compares the model against persistence on exactly the same valid sensor-targets.

**Run this notebook only after the required test target files and raw METR-LA HDF5 file are present.** It writes new files named `urbanflow_verified_test_results.csv` and `urbanflow_verified_test_audit.json`.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt

current = Path.cwd()
PROJECT_ROOT = next((p for p in [current, *current.parents] if (p / 'data/processed').exists() and (p / 'models').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate UrbanFlow project root from the notebook working directory.')

RESULTS_DIR = PROJECT_ROOT / 'results'
CSV_PATH = RESULTS_DIR / 'urbanflow_verified_test_results.csv'
AUDIT_PATH = RESULTS_DIR / 'urbanflow_verified_test_audit.json'
SCRIPT_PATH = PROJECT_ROOT / 'scripts' / 'evaluate_multihorizon_verified.py'

print('Project root:', PROJECT_ROOT)
print('Evaluation script:', SCRIPT_PATH)
print('Checkpoint will be loaded; no training will be run.')
if not SCRIPT_PATH.exists():
    raise FileNotFoundError('Missing scripts/evaluate_multihorizon_verified.py. Copy it from the UrbanFlow evaluation patch first.')

if CSV_PATH.exists() and AUDIT_PATH.exists():
    print('Existing verified evaluation outputs found; loading them without rerunning or overwriting.')
elif CSV_PATH.exists() or AUDIT_PATH.exists():
    raise FileExistsError('Only one verified output exists. Inspect results/ and move the partial file aside before rerunning.')
else:
    sys.path.insert(0, str(PROJECT_ROOT))
    from scripts.evaluate_multihorizon_verified import main as run_verified_evaluation
    run_verified_evaluation()

results = pd.read_csv(CSV_PATH)
audit = json.loads(AUDIT_PATH.read_text())
print('\nAlignment checks:')
display(pd.DataFrame(audit['alignment_checks']))
print('\nVerified test results:')
display(results)

## Compare the model with persistence

The persistence baseline predicts each sensor's latest observed input speed. Both methods are scored against the same raw-observation mask for each horizon. Positive MAE improvement means the model is better than persistence; negative means persistence is better.

In [ ]:
required_cols = {'horizon_min', 'MAE_mph', 'RMSE_mph', 'persistence_MAE_mph', 'persistence_RMSE_mph'}
missing = required_cols - set(results.columns)
if missing:
    raise ValueError(f'Evaluation results are missing expected columns: {sorted(missing)}')

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(results['horizon_min'], results['MAE_mph'], marker='o', linewidth=2, label='UrbanFlow GNN–Transformer')
ax.plot(results['horizon_min'], results['persistence_MAE_mph'], marker='s', linewidth=2, label='Persistence baseline')
ax.set_xlabel('Forecast horizon (minutes)')
ax.set_ylabel('MAE (mph)')
ax.set_title('Verified test MAE by forecast horizon')
ax.set_xticks(results['horizon_min'])
ax.grid(True, alpha=0.25)
ax.legend()
fig.tight_layout()
mae_path = RESULTS_DIR / 'urbanflow_verified_test_mae.png'
fig.savefig(mae_path, dpi=220, bbox_inches='tight')
plt.show()
print('Saved:', mae_path)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(results['horizon_min'], results['RMSE_mph'], marker='o', linewidth=2, label='UrbanFlow GNN–Transformer')
ax.plot(results['horizon_min'], results['persistence_RMSE_mph'], marker='s', linewidth=2, label='Persistence baseline')
ax.set_xlabel('Forecast horizon (minutes)')
ax.set_ylabel('RMSE (mph)')
ax.set_title('Verified test RMSE by forecast horizon')
ax.set_xticks(results['horizon_min'])
ax.grid(True, alpha=0.25)
ax.legend()
fig.tight_layout()
rmse_path = RESULTS_DIR / 'urbanflow_verified_test_rmse.png'
fig.savefig(rmse_path, dpi=220, bbox_inches='tight')
plt.show()
print('Saved:', rmse_path)

## Interpretation and reporting rules

- Report the valid observed sensor-target count for every horizon; sample counts can differ by horizon.
- Use the MAE/RMSE in this report only after reviewing `urbanflow_verified_test_audit.json` and confirming every alignment difference is zero (or within the script's strict tolerance).
- These metrics exclude zero-speed and non-finite raw observations. They are therefore not directly comparable to historical metrics calculated with a different mask.
- This evaluation does not establish that weather features improve performance; that requires a controlled ablation on the same data split and mask.
- The notebook intentionally does not overwrite the existing historical result CSVs or model checkpoint.